# Semana 1: carga y reporte WA_Crash

En este notebook se construye un flujo completo de trabajo:

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar titled-tuesday-chess-games desde Hugging Face.
3. Obtener un DataFrame de pandas y revisar su estructura.
4. Generar un reporte en pantalla y guardarlo en un archivo .txt.
    
Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con hf auth login.

# 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
while  ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/home/runner/work/PI_2026_IUD/PI_2026_IUD/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Configurar el dataset y la salida

dataset_name identifica el repositorio en Hugging Face y split="train" indica la división que se descargará. El parámetro filename hace que el DataFrame también se guarde como CSV en src/sincronicas/datos/ cuando el notebook se ejecuta desde esa carpeta.

In [2]:
hfdataset = HFDatasetLoader(
    dataset_name = "kalley/WA_Crash",
    split = "train",
    filename = "datos/WA_Crashn.csv"
)

# 3. Descargar el dataset y crear el DataFrame

El método load() descarga el dataset, lo convierte a pandas.DataFrame y lo asigna a la variable df. La expresión df.head() muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = hfdataset.load()

Generating train split:   0%|          | 0/127102 [00:00<?, ? examples/s]

Generating train split:  39%|███▉      | 50000/127102 [00:00<00:00, 416542.10 examples/s]

Generating train split:  87%|████████▋ | 110000/127102 [00:00<00:00, 449974.22 examples/s]

Generating train split: 100%|██████████| 127102/127102 [00:00<00:00, 445797.63 examples/s]

In [4]:
df.head(5)

,X,Y,ACC_ID,ROAD_NO,ROAD_NAME,COMMON_ROAD_NAME,CWAY,SLK,INTERSECTION_NO,INTERSECTION_DESC,...,EVENT_NATURE,EVENT_TYPE,TOTAL_BIKE_INVOLVED,TOTAL_TRUCK_INVOLVED,TOTAL_HEAVY_TRUCK_INVOLVED,TOTAL_MOTOR_CYCLE_INVOLVED,TOTAL_OTHER_VEHICLES_INVOLVED,TOTAL_PEDESTRIANS_INVOLVED,OBJECTID,GlobalID
0,115.907761,-31.966608,10972646,1290190,Cookham Rd,Cookham Rd,S,0.76,NaN,None,...,Rear End,Involving Parking,0,0,0,0,2,0,53073718,{DFA6DABF-C7F2-429F-A9FA-4E7D332F5320}
1,115.789670,-32.145711,10972652,H002,Melville Mandurah Hwy,Rockingham Rd,L,11.15,13935.0,Melville Mandurah Hwy & Cockburn Rd,...,Rear End,None,0,0,1,0,1,0,53073719,{025E0A45-51FE-48CB-AC07-CF059C72A92E}
2,115.980692,-31.993529,10972658,H018,Roe Hwy,Roe Hwy,L,27.78,NaN,None,...,Rear End,None,0,0,1,0,1,0,53073720,{776D4032-F18E-4FCE-8D9C-54A47649A164}
3,115.808775,-31.777105,10972671,H035,Wanneroo Rd,Wanneroo Rd,L,18.92,11972.0,Wanneroo Rd & East Rd,...,Rear End,None,0,0,0,0,2,0,53073721,{2203115F-5D82-454E-9700-9789BCF0A953}
4,115.741921,-31.826004,10972687,1318348,Southside Dr,Southside Dr,S,0.51,NaN,None,...,None,None,0,0,0,0,2,0,53073722,{769836B0-91DB-4477-9EA4-8498B75FEEFB}


# 4. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [5]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 127102
Columnas: 26
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127102 entries, 0 to 127101
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   X                              127102 non-null  float64
 1   Y                              127102 non-null  float64
 2   ACC_ID                         127102 non-null  int64  
 3   ROAD_NO                        127102 non-null  object 
 4   ROAD_NAME                      127102 non-null  object 
 5   COMMON_ROAD_NAME               127102 non-null  object 
 6   CWAY                           127102 non-null  object 
 7   SLK                            127102 non-null  float64
 8   INTERSECTION_NO                68123 non-null   float64
 9   INTERSECTION_DESC              68123 non-null   object 
 10  LONGITUDE                      127102 non-null  float64
 11  LATITUDE                       127102 non-null  float64
 12  CRA

# 5. Transformaciones del DataFrame

Para cumplir con la rúbrica de evaluación, se aplican cuatro transformaciones secuenciales al DataFrame antes de exportar el reporte final:

1. **Filtrado**: Se genera un filtrado 
2. **Eliminación de columnas repetidas**: Eliminar las columnas X Y, ya que actualmente se cuenta con las columnas Longitud y Latitud las cuales expresan los mismos valores de referencia con relación a la ubicación.
3. **Tratamiento de nulos**: Imputación de valores nulos en las columnas CRASH_TIME, EVENT_NATURE y EVENT_TYPE.
4. **Renombrado**: Renombrar las columnas de nombre CWAY a CARRIEAGEWAY y SLK a STRAIGHT_LINE_KILOMETRE

# Transformación 1: Filtrado de datos a partir de una columna

Se realiza el filtrado de datos sobre la columna EVENT_NATURE cuando el valor es igual a Rear End.

In [6]:
df_transformado = df.copy()

dt_filtrado = df_transformado[df_transformado['EVENT_NATURE'] == 'Rear End']

#print(dt_filtrado)
dt_filtrado.head()

,X,Y,ACC_ID,ROAD_NO,ROAD_NAME,COMMON_ROAD_NAME,CWAY,SLK,INTERSECTION_NO,INTERSECTION_DESC,...,EVENT_NATURE,EVENT_TYPE,TOTAL_BIKE_INVOLVED,TOTAL_TRUCK_INVOLVED,TOTAL_HEAVY_TRUCK_INVOLVED,TOTAL_MOTOR_CYCLE_INVOLVED,TOTAL_OTHER_VEHICLES_INVOLVED,TOTAL_PEDESTRIANS_INVOLVED,OBJECTID,GlobalID
0,115.907761,-31.966608,10972646,1290190,Cookham Rd,Cookham Rd,S,0.76,NaN,None,...,Rear End,Involving Parking,0,0,0,0,2,0,53073718,{DFA6DABF-C7F2-429F-A9FA-4E7D332F5320}
1,115.789670,-32.145711,10972652,H002,Melville Mandurah Hwy,Rockingham Rd,L,11.15,13935.0,Melville Mandurah Hwy & Cockburn Rd,...,Rear End,None,0,0,1,0,1,0,53073719,{025E0A45-51FE-48CB-AC07-CF059C72A92E}
2,115.980692,-31.993529,10972658,H018,Roe Hwy,Roe Hwy,L,27.78,NaN,None,...,Rear End,None,0,0,1,0,1,0,53073720,{776D4032-F18E-4FCE-8D9C-54A47649A164}
3,115.808775,-31.777105,10972671,H035,Wanneroo Rd,Wanneroo Rd,L,18.92,11972.0,Wanneroo Rd & East Rd,...,Rear End,None,0,0,0,0,2,0,53073721,{2203115F-5D82-454E-9700-9789BCF0A953}
5,116.031149,-31.670670,10972693,H006,Great Northern Hwy,Great Northern Hwy,S,25.43,NaN,None,...,Rear End,None,0,0,0,0,2,0,53073723,{67ABE5F0-D3C6-4D32-9906-4B873F051EA3}


# Transformación 2: Eliminación de columnas repetidas

Eliminar las columnas X Y, ya que actualmente se cuenta con las columnas Longitud y Latitud las cuales expresan los mismos valores de referencia con relación a la ubicación.

In [7]:
df_filtrado = df_transformado.drop(columns=["X", "Y"])

df_filtrado.head()

,ACC_ID,ROAD_NO,ROAD_NAME,COMMON_ROAD_NAME,CWAY,SLK,INTERSECTION_NO,INTERSECTION_DESC,LONGITUDE,LATITUDE,...,EVENT_NATURE,EVENT_TYPE,TOTAL_BIKE_INVOLVED,TOTAL_TRUCK_INVOLVED,TOTAL_HEAVY_TRUCK_INVOLVED,TOTAL_MOTOR_CYCLE_INVOLVED,TOTAL_OTHER_VEHICLES_INVOLVED,TOTAL_PEDESTRIANS_INVOLVED,OBJECTID,GlobalID
0,10972646,1290190,Cookham Rd,Cookham Rd,S,0.76,NaN,None,115.907761,-31.966608,...,Rear End,Involving Parking,0,0,0,0,2,0,53073718,{DFA6DABF-C7F2-429F-A9FA-4E7D332F5320}
1,10972652,H002,Melville Mandurah Hwy,Rockingham Rd,L,11.15,13935.0,Melville Mandurah Hwy & Cockburn Rd,115.789670,-32.145711,...,Rear End,None,0,0,1,0,1,0,53073719,{025E0A45-51FE-48CB-AC07-CF059C72A92E}
2,10972658,H018,Roe Hwy,Roe Hwy,L,27.78,NaN,None,115.980692,-31.993529,...,Rear End,None,0,0,1,0,1,0,53073720,{776D4032-F18E-4FCE-8D9C-54A47649A164}
3,10972671,H035,Wanneroo Rd,Wanneroo Rd,L,18.92,11972.0,Wanneroo Rd & East Rd,115.808775,-31.777105,...,Rear End,None,0,0,0,0,2,0,53073721,{2203115F-5D82-454E-9700-9789BCF0A953}
4,10972687,1318348,Southside Dr,Southside Dr,S,0.51,NaN,None,115.741921,-31.826004,...,None,None,0,0,0,0,2,0,53073722,{769836B0-91DB-4477-9EA4-8498B75FEEFB}


# Transformación 3: Tratamiento de valores nulos

Imputación de valores nulos en las columnas CRASH_TIME, EVENT_NATURE, EVENT_TYPE, INTERSECTION_NO, INTERSECTION_DESC.

In [8]:
#INTERSECTION_NO
#INTERSECTION_DESC
df_filtrado['CRASH_TIME'] = df_filtrado['CRASH_TIME'].fillna(df_filtrado['CRASH_TIME'].mean())
#df_filtrado['EVENT_NATURE'] = df_filtrado['EVENT_NATURE'].fillna('NaN')
#df_filtrado['EVENT_TYPE'] = df_filtrado['EVENT_TYPE'].fillna('NaN')
#df_filtrado['INTERSECTION_NO'] = df_filtrado['INTERSECTION_NO'].fillna('NaN')
#df_filtrado['INTERSECTION_DESC'] = df_filtrado['INTERSECTION_DESC'].fillna('NaN')

print(df_transformado)

                 X          Y    ACC_ID  ROAD_NO              ROAD_NAME  \
0       115.907761 -31.966608  10972646  1290190             Cookham Rd   
1       115.789670 -32.145711  10972652     H002  Melville Mandurah Hwy   
2       115.980692 -31.993529  10972658     H018                Roe Hwy   
3       115.808775 -31.777105  10972671     H035            Wanneroo Rd   
4       115.741921 -31.826004  10972687  1318348           Southside Dr   
...            ...        ...       ...      ...                    ...   
127097  115.718191 -33.318133  10966247  2080275             Council Dr   
127098  115.891659 -32.067495  10966253     H018                Roe Hwy   
127099  115.877625 -31.850776  10966265  1090697               Beach Rd   
127100  115.867349 -31.946757  10966272  1300250              Edward St   
127101  115.797787 -32.454986  10966277     H015            Kwinana Fwy   

       COMMON_ROAD_NAME CWAY    SLK  INTERSECTION_NO  \
0            Cookham Rd    S   0.76        

# Transformación 4: Renombrado de columnas

Renombrar las columnas de nombre CWAY a CARRIEAGEWAY y SLK a STRAIGHT_LINE_KILOMETRE

In [9]:
# 4. Renombrado de una columna

df_filtrado = df_filtrado.rename(
    columns={
        "CWAY" : "CARRIEAGEWAY",
        "SLK" : "STRAIGHT_LINE_KILOMETRE"
    }
)

print(f"Dimensiones luego de aplicar las 4 transformaciones: {df.shape[0]} filas x {df.shape[1]} columnas")
df_filtrado.head()

Dimensiones luego de aplicar las 4 transformaciones: 127102 filas x 26 columnas


,ACC_ID,ROAD_NO,ROAD_NAME,COMMON_ROAD_NAME,CARRIEAGEWAY,STRAIGHT_LINE_KILOMETRE,INTERSECTION_NO,INTERSECTION_DESC,LONGITUDE,LATITUDE,...,EVENT_NATURE,EVENT_TYPE,TOTAL_BIKE_INVOLVED,TOTAL_TRUCK_INVOLVED,TOTAL_HEAVY_TRUCK_INVOLVED,TOTAL_MOTOR_CYCLE_INVOLVED,TOTAL_OTHER_VEHICLES_INVOLVED,TOTAL_PEDESTRIANS_INVOLVED,OBJECTID,GlobalID
0,10972646,1290190,Cookham Rd,Cookham Rd,S,0.76,NaN,None,115.907761,-31.966608,...,Rear End,Involving Parking,0,0,0,0,2,0,53073718,{DFA6DABF-C7F2-429F-A9FA-4E7D332F5320}
1,10972652,H002,Melville Mandurah Hwy,Rockingham Rd,L,11.15,13935.0,Melville Mandurah Hwy & Cockburn Rd,115.789670,-32.145711,...,Rear End,None,0,0,1,0,1,0,53073719,{025E0A45-51FE-48CB-AC07-CF059C72A92E}
2,10972658,H018,Roe Hwy,Roe Hwy,L,27.78,NaN,None,115.980692,-31.993529,...,Rear End,None,0,0,1,0,1,0,53073720,{776D4032-F18E-4FCE-8D9C-54A47649A164}
3,10972671,H035,Wanneroo Rd,Wanneroo Rd,L,18.92,11972.0,Wanneroo Rd & East Rd,115.808775,-31.777105,...,Rear End,None,0,0,0,0,2,0,53073721,{2203115F-5D82-454E-9700-9789BCF0A953}
4,10972687,1318348,Southside Dr,Southside Dr,S,0.51,NaN,None,115.741921,-31.826004,...,None,None,0,0,0,0,2,0,53073722,{769836B0-91DB-4477-9EA4-8498B75FEEFB}


In [10]:
from  clases_po import Reportes

# 6 Generar el resumen del DataFrame transformado

La clase Reportes recibe el DataFrame final, muestra sus dimensiones, tipos de columnas, valores nulos y estadisticas numericas.

In [11]:
reportes = Reportes()
texto = reportes.generar_reporte(df_filtrado)

print(texto)

REPORTE DEL DATAFRAME
Filas: 127102
Columnas: 24
Duplicados: 0

Columnas y tipos:
- ACC_ID: int64
- ROAD_NO: object
- ROAD_NAME: object
- COMMON_ROAD_NAME: object
- CARRIEAGEWAY: object
- STRAIGHT_LINE_KILOMETRE: float64
- INTERSECTION_NO: float64
- INTERSECTION_DESC: object
- LONGITUDE: float64
- LATITUDE: float64
- CRASH_DATE: object
- CRASH_TIME: float64
- ACCIDENT_TYPE: object
- SEVERITY: object
- EVENT_NATURE: object
- EVENT_TYPE: object
- TOTAL_BIKE_INVOLVED: int64
- TOTAL_TRUCK_INVOLVED: int64
- TOTAL_HEAVY_TRUCK_INVOLVED: int64
- TOTAL_MOTOR_CYCLE_INVOLVED: int64
- TOTAL_OTHER_VEHICLES_INVOLVED: int64
- TOTAL_PEDESTRIANS_INVOLVED: int64
- OBJECTID: int64
- GlobalID: object

Valores nulos por columna:
- ACC_ID: 0
- ROAD_NO: 0
- ROAD_NAME: 0
- COMMON_ROAD_NAME: 0
- CARRIEAGEWAY: 0
- STRAIGHT_LINE_KILOMETRE: 0
- INTERSECTION_NO: 58979
- INTERSECTION_DESC: 58979
- LONGITUDE: 0
- LATITUDE: 0
- CRASH_DATE: 0
- CRASH_TIME: 0
- ACCIDENT_TYPE: 0
- SEVERITY: 0
- EVENT_NATURE: 7699
- EVEN

# 7. Guardar el resumen en un archivo .txt

El reporte corresponde al DataFrame transformado se guardado en la ruta src/sincronicas/reportes/WA_Crash.txt.

In [12]:
ruta = reportes.guardar_reporte(
    df_filtrado,
    "reportes/WA_Crash.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/WA_Crash.txt
